# 02 - Model Predictive Control

This notebook starts from the LQR idea and asks a practical question: what changes when the actuator and the predicted states must satisfy constraints?

The main message is visible in two closed-loop simulations:
- a short horizon needs a good terminal cost to behave like the infinite-horizon LQR;
- clipping LQR is not the same as solving a constrained optimal control problem.


In [ ]:
%matplotlib inline


## The Teaching Plant

We use the 2D discrete double integrator from the LQR notebook.

What are we comparing?
- position and velocity states;
- one scalar acceleration input;
- the same quadratic stage cost in every experiment.

The infinite-horizon LQR gain comes from the DARE. Later we will use `P_inf` as a terminal cost.


In [ ]:
import numpy as np
from scipy.linalg import solve_discrete_are

np.set_printoptions(precision=3, suppress=True)

A = np.array([[1.0, 1.0],
              [0.0, 1.0]])
B = np.array([[0.0],
              [1.0]])
Q = np.eye(2)
R = np.array([[1.0]])

nx = A.shape[0]
nu = B.shape[1]

P_inf = solve_discrete_are(A, B, Q, R)
K_inf = np.linalg.solve(R + B.T @ P_inf @ B, B.T @ P_inf @ A)

print("Infinite-horizon LQR gain K_inf =", K_inf)
print("DARE terminal matrix P_inf =")
print(P_inf)


## A Short Horizon Has A Missing Tail

What are we comparing?
- receding-horizon controllers with the same short horizon;
- only the terminal cost changes.

What should students observe?
- no or weak terminal cost makes the controller more short-sighted;
- better terminal approximations move the closed loop toward LQR;
- using `P_inf` recovers infinite-horizon LQR in the unconstrained case.

First we make a few possible terminal matrices. The Riccati update is written directly in the loop so the update is visible.


In [ ]:
terminal_horizon = 3
simulation_steps = 14
x_initial = np.array([5.0, 0.0])

P_tail_1 = np.zeros((nx, nx))
for tail_step in range(1):
    K_tail = np.linalg.solve(R + B.T @ P_tail_1 @ B, B.T @ P_tail_1 @ A)
    P_tail_1 = Q + A.T @ P_tail_1 @ (A - B @ K_tail)

P_tail_2 = np.zeros((nx, nx))
for tail_step in range(2):
    K_tail = np.linalg.solve(R + B.T @ P_tail_2 @ B, B.T @ P_tail_2 @ A)
    P_tail_2 = Q + A.T @ P_tail_2 @ (A - B @ K_tail)

P_tail_20 = np.zeros((nx, nx))
for tail_step in range(20):
    K_tail = np.linalg.solve(R + B.T @ P_tail_20 @ B, B.T @ P_tail_20 @ A)
    P_tail_20 = Q + A.T @ P_tail_20 @ (A - B @ K_tail)

terminal_costs = {
    "Vf = 0": np.zeros((nx, nx)),
    "Vf = Q": Q,
    "Vf = P_tail_1": P_tail_1,
    "Vf = P_tail_2": P_tail_2,
    "Vf = P_tail_20": P_tail_20,
    "Vf = P_inf": P_inf,
}

for name, P_terminal in terminal_costs.items():
    print(name)
    print(P_terminal)


Now we simulate the receding-horizon controllers. For each terminal cost we compute the first finite-horizon gain by running the backward Riccati recursion for `terminal_horizon` steps.

In this unconstrained, time-invariant LQR example the same first gain is obtained at every sampling time. That is why the closed-loop simulation below can use one gain per terminal cost.


In [ ]:
import matplotlib.pyplot as plt

runs = {}

for name, P_terminal in terminal_costs.items():
    P_next = P_terminal.copy()
    for backward_step in range(terminal_horizon):
        K_first = np.linalg.solve(R + B.T @ P_next @ B, B.T @ P_next @ A)
        P_next = Q + A.T @ P_next @ (A - B @ K_first)

    X = np.zeros((simulation_steps + 1, nx))
    U = np.zeros(simulation_steps)
    X[0] = x_initial

    for k in range(simulation_steps):
        U[k] = -float((K_first @ X[k]).item())
        X[k + 1] = A @ X[k] + B[:, 0] * U[k]

    runs[name] = (X, U)

X_lqr_reference = np.zeros((simulation_steps + 1, nx))
U_lqr_reference = np.zeros(simulation_steps)
X_lqr_reference[0] = x_initial

for k in range(simulation_steps):
    U_lqr_reference[k] = -float((K_inf @ X_lqr_reference[k]).item())
    X_lqr_reference[k + 1] = A @ X_lqr_reference[k] + B[:, 0] * U_lqr_reference[k]

runs["infinite-horizon LQR"] = (X_lqr_reference, U_lqr_reference)

time = np.arange(simulation_steps + 1)
fig, axes = plt.subplots(3, 1, figsize=(8.0, 7.0), sharex=True)

for label, (X, U) in runs.items():
    linewidth = 2.4 if label in ["Vf = P_inf", "infinite-horizon LQR"] else 1.4
    linestyle = "--" if label == "infinite-horizon LQR" else "-"
    axes[0].plot(time, X[:, 0], label=label, linewidth=linewidth, linestyle=linestyle)
    axes[1].plot(time, X[:, 1], label=label, linewidth=linewidth, linestyle=linestyle)
    axes[2].step(time[:-1], U, where="post", label=label, linewidth=linewidth, linestyle=linestyle)

axes[0].set_ylabel("position")
axes[1].set_ylabel("velocity")
axes[2].set_ylabel("input")
axes[2].set_xlabel("time step")
for ax in axes:
    ax.grid(True, alpha=0.25)
axes[0].legend(loc="best", ncols=2)
fig.tight_layout()


### Interpretation

- The controller with `Vf = 0` sees only three future steps, so its first moves are visibly different.
- One or two Riccati tail steps already improve the behavior.
- `P_tail_20` is almost indistinguishable from `P_inf`.
- With `P_inf`, the unconstrained receding-horizon controller overlays the infinite-horizon LQR reference.


## Build The Finite-Horizon QP By Hand

Now constraints enter. We condense the prediction equations into

`X = Sx x0 + Su U`

where `U` is the whole planned input sequence.

Control message: MPC optimizes a sequence, not just the current clipped input.


### Prediction Matrices

The stacked state vector is

`X = [x1, x2, ..., xN]`

and the stacked input vector is

`U = [u0, u1, ..., u_{N-1}]`.

The next cell builds `Sx` and `Su` directly from powers of `A`.


In [ ]:
from scipy.linalg import block_diag
from scipy.optimize import Bounds, LinearConstraint, minimize

constrained_horizon = 12
constrained_steps = 30
u_limit = 0.5
position_min = 0.0
x0 = np.array([0.0, 4.0])

Sx = np.zeros((constrained_horizon * nx, nx))
Su = np.zeros((constrained_horizon * nx, constrained_horizon * nu))

for i in range(constrained_horizon):
    row = slice(i * nx, (i + 1) * nx)
    Sx[row, :] = np.linalg.matrix_power(A, i + 1)

    for j in range(i + 1):
        col = slice(j * nu, (j + 1) * nu)
        Su[row, col] = np.linalg.matrix_power(A, i - j) @ B

print("Sx shape:", Sx.shape)
print("Su shape:", Su.shape)


### Small Matrix Display Example

The production horizon above is useful for simulation, but it is too wide for a readable board display. The next cell rebuilds the same matrices for `N_display = 3` so the structure is visible.


In [ ]:
from IPython.display import Markdown, display

N_display = 3
Sx_display = np.zeros((N_display * nx, nx))
Su_display = np.zeros((N_display * nx, N_display * nu))

for i in range(N_display):
    row = slice(i * nx, (i + 1) * nx)
    Sx_display[row, :] = np.linalg.matrix_power(A, i + 1)

    for j in range(i + 1):
        col = slice(j * nu, (j + 1) * nu)
        Su_display[row, col] = np.linalg.matrix_power(A, i - j) @ B

Qbar_display = block_diag(*([Q] * (N_display - 1) + [Q + P_inf]))
Rbar_display = block_diag(*([R] * N_display))

x0_display = x0
predicted_free_display = Sx_display @ x0_display
H_display = Su_display.T @ Qbar_display @ Su_display + Rbar_display
f_display = Su_display.T @ Qbar_display @ predicted_free_display

state_rows = [f"x{k}_{part}" for k in range(1, N_display + 1) for part in ["pos", "vel"]]
state_cols = ["x0_pos", "x0_vel"]
input_cols = [f"u{k}" for k in range(N_display)]


def show_matrix(name, values, row_labels, column_labels):
    values = np.round(values, 3)
    lines = [f"**{name}**", ""]
    lines.append("| | " + " | ".join(column_labels) + " |")
    lines.append("|" + "|".join(["---"] * (len(column_labels) + 1)) + "|")

    for row_label, row_values in zip(row_labels, values):
        entries = []
        for value in row_values:
            if abs(value) < 1e-12:
                value = 0.0
            entries.append(f"{value:.3g}")
        lines.append(f"| {row_label} | " + " | ".join(entries) + " |")

    display(Markdown(chr(10).join(lines)))


show_matrix("Sx_display", Sx_display, state_rows, state_cols)
show_matrix("Su_display", Su_display, state_rows, input_cols)
show_matrix("Qbar_display", Qbar_display, state_rows, state_rows)
show_matrix("Rbar_display", Rbar_display, input_cols, input_cols)
show_matrix("H_display", H_display, input_cols, input_cols)
show_matrix("f_display", f_display.reshape(-1, 1), input_cols, ["linear term"])


The display shows the objects that a condensed QP solver receives.

- `Sx` propagates the initial state through the horizon.
- `Su` shows how each planned input affects each future state.
- `Qbar` and `Rbar` collect the state and input cost weights.
- `H` is the quadratic cost matrix of the condensed QP, and `f` is the linear term for this particular `x0`.
- CasADi later automates this construction, but it is solving the same kind of QP.


### Condensed Cost

For the first constrained example we use `P_inf` as the terminal cost.

The term `x0.T @ Q @ x0` is constant with respect to `U`, so it is not included in the optimizer. The matrix `Qbar` weights `x1` through `xN`; the last predicted state receives both the stage cost `Q` and the terminal cost `P_terminal`.


In [ ]:
P_terminal = P_inf

Qbar = block_diag(*([Q] * (constrained_horizon - 1) + [Q + P_terminal]))
Rbar = block_diag(*([R] * constrained_horizon))

H = Su.T @ Qbar @ Su + Rbar
predicted_free = Sx @ x0
f = Su.T @ Qbar @ predicted_free

print("Qbar shape:", Qbar.shape)
print("Rbar shape:", Rbar.shape)
print("H shape:", H.shape)
print("f shape:", f.shape)


### First Unconstrained QP

Before adding constraints, the condensed quadratic problem is solved by the linear system

`H U = -f`.

This is the planned input sequence that would be optimal if the actuator and state were unlimited.


In [ ]:
U_unconstrained = -np.linalg.solve(H, f)
X_unconstrained_plan = (Sx @ x0 + Su @ U_unconstrained).reshape(constrained_horizon, nx)

print("first unconstrained input =", U_unconstrained[0])
print("largest unconstrained |u| =", np.max(np.abs(U_unconstrained)))
print("minimum planned position =", np.min(X_unconstrained_plan[:, 0]))


### Add The Input Bound

The actuator limit is

`-u_limit <= uk <= u_limit`.

This still does not protect the future state constraint; it only clips the feasible range of the planned inputs.


In [ ]:
input_bounds = Bounds(
    -u_limit * np.ones(constrained_horizon),
    u_limit * np.ones(constrained_horizon),
)

input_bounded_result = minimize(
    lambda U: 0.5 * U @ H @ U + f @ U,
    np.zeros(constrained_horizon),
    jac=lambda U: H @ U + f,
    bounds=input_bounds,
    method="SLSQP",
    options={"ftol": 1e-9, "maxiter": 300},
)

U_input_bounded = input_bounded_result.x
X_input_bounded_plan = (Sx @ x0 + Su @ U_input_bounded).reshape(constrained_horizon, nx)

print("input-bounded QP success:", input_bounded_result.success)
print("first input-bounded input =", U_input_bounded[0])
print("minimum planned position =", np.min(X_input_bounded_plan[:, 0]))


### Add The State Constraint

The wall constraint is

`position >= 0`.

Since every predicted position is inside `X = Sx x0 + Su U`, the constraint becomes a linear inequality in the planned input sequence `U`.


In [ ]:
position_rows = Su[0::nx, :]
position_free = predicted_free[0::nx]
position_lower = position_min - position_free
position_upper = np.full(constrained_horizon, np.inf)
wall_constraint = LinearConstraint(position_rows, position_lower, position_upper)

constrained_result = minimize(
    lambda U: 0.5 * U @ H @ U + f @ U,
    np.zeros(constrained_horizon),
    jac=lambda U: H @ U + f,
    bounds=input_bounds,
    constraints=[wall_constraint],
    method="SLSQP",
    options={"ftol": 1e-9, "maxiter": 300},
)

planned_U = constrained_result.x
planned_X = (Sx @ x0 + Su @ planned_U).reshape(constrained_horizon, nx)

print("fully constrained QP success:", constrained_result.success)
print("first constrained input =", planned_U[0])
print("minimum planned position =", np.min(planned_X[:, 0]))


## Saturated LQR Versus Constrained MPC

What are we comparing?
- unconstrained LQR;
- the same LQR with immediate input clipping;
- MPC with both input bounds and a predicted position constraint.

The position constraint is `position >= 0`. The initial state has large positive velocity, so every controller first brakes. The difference appears later: saturated LQR respects the input bound but does not plan the future wall crossing.


In [ ]:
X_lqr = np.zeros((constrained_steps + 1, nx))
U_lqr = np.zeros(constrained_steps)
X_lqr[0] = x0

for k in range(constrained_steps):
    U_lqr[k] = -float((K_inf @ X_lqr[k]).item())
    X_lqr[k + 1] = A @ X_lqr[k] + B[:, 0] * U_lqr[k]

X_sat = np.zeros((constrained_steps + 1, nx))
U_sat = np.zeros(constrained_steps)
X_sat[0] = x0

for k in range(constrained_steps):
    raw_lqr_input = -float((K_inf @ X_sat[k]).item())
    U_sat[k] = np.clip(raw_lqr_input, -u_limit, u_limit)
    X_sat[k + 1] = A @ X_sat[k] + B[:, 0] * U_sat[k]

X_mpc = np.zeros((constrained_steps + 1, nx))
U_mpc = np.zeros(constrained_steps)
mpc_success = []
X_mpc[0] = x0

for k in range(constrained_steps):
    x_now = X_mpc[k]
    predicted_free = Sx @ x_now
    f = Su.T @ Qbar @ predicted_free

    position_free = predicted_free[0::nx]
    position_lower = position_min - position_free
    wall_constraint = LinearConstraint(position_rows, position_lower, position_upper)

    result = minimize(
        lambda U: 0.5 * U @ H @ U + f @ U,
        np.zeros(constrained_horizon),
        jac=lambda U: H @ U + f,
        bounds=input_bounds,
        constraints=[wall_constraint],
        method="SLSQP",
        options={"ftol": 1e-9, "maxiter": 300},
    )

    mpc_success.append(result.success)
    if not result.success:
        raise RuntimeError(f"MPC QP failed at step {k}: {result.message}")

    U_mpc[k] = result.x[0]
    X_mpc[k + 1] = A @ X_mpc[k] + B[:, 0] * U_mpc[k]

constrained_runs = {
    "unconstrained LQR": (X_lqr, U_lqr),
    "saturated LQR": (X_sat, U_sat),
    "constrained MPC": (X_mpc, U_mpc),
}

print("MPC solver successes:", sum(mpc_success), "of", len(mpc_success))
print("minimum position:", {name: float(np.min(X[:, 0])) for name, (X, _) in constrained_runs.items()})
print("maximum |u|:", {name: float(np.max(np.abs(U))) for name, (_, U) in constrained_runs.items()})


In [ ]:
time = np.arange(constrained_steps + 1)
fig, axes = plt.subplots(4, 1, figsize=(8.0, 8.5), sharex=True)

for label, (X, U) in constrained_runs.items():
    linewidth = 2.2 if label == "constrained MPC" else 1.5
    axes[0].plot(time, X[:, 0], label=label, linewidth=linewidth)
    axes[1].plot(time, X[:, 1], label=label, linewidth=linewidth)
    axes[2].step(time[:-1], U, where="post", label=label, linewidth=linewidth)
    axes[3].plot(time, X[:, 0] - position_min, label=label, linewidth=linewidth)

axes[0].axhline(position_min, color="k", linestyle="--", linewidth=0.9)
axes[2].axhline(u_limit, color="k", linestyle="--", linewidth=0.9)
axes[2].axhline(-u_limit, color="k", linestyle="--", linewidth=0.9)
axes[3].axhline(0.0, color="k", linewidth=0.9)

axes[0].set_ylabel("position")
axes[1].set_ylabel("velocity")
axes[2].set_ylabel("input")
axes[3].set_ylabel("wall margin")
axes[3].set_xlabel("time step")
for ax in axes:
    ax.grid(True, alpha=0.25)
axes[0].legend(loc="best")
fig.tight_layout()


### Interpretation

- Unconstrained LQR asks for inputs outside the actuator limit.
- Saturated LQR clips those inputs, but it later crosses the wall because it never optimized the future trajectory under the wall constraint.
- MPC brakes and then plans the recovery sequence so the predicted positions stay feasible.
- Once constraints are active, the Riccati/LQR gain is no longer enough information.


## Inspect One MPC Plan

Before simulating, MPC predicts a whole sequence. Here we reuse the first fully constrained problem and plot the planned position, velocity, and input.


In [ ]:
planned_time = np.arange(1, constrained_horizon + 1)
input_time = np.arange(constrained_horizon)

fig, axes = plt.subplots(3, 1, figsize=(8.0, 6.8), sharex=False)
axes[0].plot(planned_time, planned_X[:, 0], marker="o")
axes[0].axhline(position_min, color="k", linestyle="--", linewidth=0.9)
axes[1].plot(planned_time, planned_X[:, 1], marker="o")
axes[2].step(input_time, planned_U, where="post")
axes[2].plot(input_time, planned_U, "o")
axes[2].axhline(u_limit, color="k", linestyle="--", linewidth=0.9)
axes[2].axhline(-u_limit, color="k", linestyle="--", linewidth=0.9)
axes[0].set_ylabel("planned position")
axes[1].set_ylabel("planned velocity")
axes[2].set_ylabel("planned input")
axes[2].set_xlabel("prediction step")
for ax in axes:
    ax.grid(True, alpha=0.25)
fig.tight_layout()


## CasADi At The End

Now that the hand-built QP is visible, CasADi is useful as a formulation tool. It solves the same constrained MPC problem: double-integrator dynamics, input bounds, and `position >= 0` along the prediction horizon.


In [ ]:
import casadi as ca

A_ca = ca.DM(A)
B_ca = ca.DM(B)
Q_ca = ca.DM(Q)
R_ca = ca.DM(R)
P_terminal_ca = ca.DM(P_terminal)

opti = ca.Opti()
X_ca = opti.variable(nx, constrained_horizon + 1)
U_ca = opti.variable(nu, constrained_horizon)

cost = 0
opti.subject_to(X_ca[:, 0] == ca.DM(x0))

for k in range(constrained_horizon):
    xk = X_ca[:, k]
    uk = U_ca[:, k]

    opti.subject_to(X_ca[:, k + 1] == A_ca @ xk + B_ca @ uk)
    opti.subject_to(opti.bounded(-u_limit, uk, u_limit))
    opti.subject_to(X_ca[0, k + 1] >= position_min)

    cost += ca.mtimes([xk.T, Q_ca, xk])
    cost += ca.mtimes([uk.T, R_ca, uk])

terminal_state = X_ca[:, constrained_horizon]
cost += ca.mtimes([terminal_state.T, P_terminal_ca, terminal_state])

opti.minimize(cost)
opti.solver("ipopt", {"print_time": False}, {"print_level": 0})
solution = opti.solve()

casadi_first_u = float(solution.value(U_ca[0, 0]))

print("first input from hand QP  =", U_mpc[0])
print("first input from CasADi QP =", casadi_first_u)
